# In this notebook, we simulate an approximation of the dynamics of a system coupled to a bath in the Caldeira-Leggett model

We follow the work of Breuer & Petrucione, and simulate a master equation

## Classical simulation in the linear case

For the system (an oscillator), we set the frequency in the middle of the window $\omega_0=3.5$ GHz and we choose the same capacitance order of magnitude as the one given for the mode $b$ in Lescanne, Raphaël, Marius Villiers, Théau Peronnin, Alain Sarlette, Matthieu Delbecq, Benjamin Huard, Takis Kontos, Mazyar Mirrahimi, et Zaki Leghtas. « Exponential Suppression of Bit-Flips in a Qubit Encoded in an Oscillator ». Nature Physics 16, nᵒ 5 (mai 2020): 509‑13. https://doi.org/10.1038/s41567-020-0824-x, so $E_C=73.5$ MHz and

\begin{equation}
    1/C=2E_C/e^2=2\times2\pi\frac{E_C}{h}\frac{\hbar}{e^2}=0.147\times2\pi GHz,
\end{equation}
where we set $\hbar=1$ and $e$=1. We also set $\omega/2\pi=4.886$ GHz.



Then, following Jeffrey, Evan, Daniel Sank, J. Y. Mutus, T. C. White, J. Kelly, R. Barends, Y. Chen, et al. « Fast Accurate State Measurement with Superconducting Qubits ». Physical Review Letters 112, nᵒ 19 (15 mai 2014): 190504. https://doi.org/10.1103/PhysRevLett.112.190504, we set the dissipation rate of the bath to $\kappa=1/20$ ns.

In [ ]:
using LinearAlgebra, FFTW
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/Plot_setting.jl");

In [ ]:
C_inv=0.147 * 2pi  #ħ/e^2 GHz 
ω=4.886 * 2pi
κ=1/20

M=[0 C_inv
-ω^2/C_inv -2κ]

In [ ]:
X_0=[1,1]
t=0:0.01:1/κ
X_t=[exp(M*t[i])*X_0 for i in 1:length(t)]
ϕ_t=getindex.(X_t,1)
q_t=getindex.(X_t,2)

In [ ]:
plot(t, ϕ_t, label="ϕ", legend=:topright) #xlims=(0,10)
plot!(t, q_t, label="q", xlabel="t [ns]", legend=:topright)
title!("Caldeira-Leggett master equation")

In [ ]:
Ω=sqrt(ω^2-κ^2)
q_analytic= @. exp(-κ*t)*(X_0[2]*cos(Ω*t) - (κ*X_0[2]+ω^2*X_0[1]/C_inv)*sin(Ω*t)/Ω)
ϕ_analytic= @. exp(-κ*t)*(X_0[1]*cos(Ω*t) + (X_0[2]*C_inv+κ*X_0[1])*sin(Ω*t)/Ω)

In [ ]:
2κ^2/ω^2

In [ ]:
plot(t, ϕ_analytic, label="ϕ", legend=:topright)
plot!(t, q_analytic, label="q", xlabel="t [ns]", legend=:topright)
title!("Caldeira-Leggett master equation")

In [ ]:
plot(ϕ_analytic, q_analytic, label="", xlabel="ϕ", ylabel="q", legend=:topright, linewidth=.2)
savefig("Caldeira-Leggett_master_equation_linear.pdf")

## Approximation

The Caldeira-Leggett master equation can be approximated by a Lindblad equation if
$$
\kappa\ll\omega ;~~~~~~~ q_0\approx \phi_0
$$

In [ ]:
q_approx= @. exp(-κ*t)*(X_0[2]*cos(ω*t) - ω*X_0[1]/C_inv*sin(ω*t))
ϕ_approx= @. exp(-κ*t)*(X_0[1]*cos(ω*t) + X_0[2]*C_inv*sin(ω*t)/ω)

In [ ]:
plot(t, ϕ_approx, label="ϕ", legend=:topright)
plot!(t, q_approx, label="q", xlabel="t [ns]", legend=:topright)
title!("Lindblad master equation")

## Classical simulation in the non-linear case

We consider a circuit with a capacitor and a Josephson junction, coupled to the environment.

In [ ]:
using DifferentialEquations

In [ ]:
# Definition of the second order ODE
function damped_non_linear(ddu, du, u, p, t)
    E_J = 90 *2pi /2 # ħ * GHz
    ϕ_quantum = 1/2 # ħ/e 
    ddu[1] = -2 * κ * du[1] - 4 * ϕ_quantum * E_J * C_inv *sin(u[1] / ϕ_quantum)
end

# Initial conditions
ϕ_0 = [1.]
ϕ_dot_0 = [1.]

κ = 1 / 20 # GHZ Damping rate
C_inv = 0.147 * 2 * pi # ħ/e^2 in GHz

E_J = 90 *2pi /2  # ħ * GHz
ϕ_quantum = 1/2 # ħ/e 
ω_nl = sqrt(4*C_inv*E_J)
Z_nl=C_inv/ω_nl

tspan = (0.0, 4.0/κ) # Time span for the simulation
dt = 1 / (20 * ω_nl) # Time step for the simulation

prob = SecondOrderODEProblem(damped_non_linear, ϕ_0, ϕ_dot_0, tspan)
sol = solve(prob, VelocityVerlet(), dt=dt)

# Extract solution arrays
ϕ = [u[1] for u in sol.u]
ϕ_dot = [u[2] for u in sol.u]
q = ϕ_dot / C_inv;


In [ ]:
sqrt(4 * E_J * C_inv)/2pi

In [ ]:
plot(sol.t, ϕ, label="ϕ", legend=:topright, linewidth=.2)
plot!(sol.t, ϕ_dot, label="ϕ_dot", xlabel="t [ns]", legend=:topright)
plot!(xlims=(0, 2))

In [ ]:
plot(ϕ, q, legend=:topright, xlabel="ϕ", ylabel="q", linewidth=.2, label="")
#savefig("Caldeira-Leggett_master_equation_non_linear.pdf")

In [ ]:
F = fftshift(fft(ϕ))
fs = length(sol.t)/(maximum(sol.t)-minimum(sol.t)) # sampling frequency
freqs = fftshift(fftfreq(length(sol.t), fs))

# plots 
time_domain = plot(sol.t,ϕ, title = "Signal", label="")
freq_domain = plot(freqs, abs.(F), title = "Spectrum", xlim=(3, 6), label="") 
f_max=abs(freqs[argmax(abs.(F))])
vline!(freq_domain, [f_max], label = "")
plot(time_domain, freq_domain, layout = (2,1))

## Classical simulation in the non-linear case with drive

In [ ]:
# Definition of the second order ODE
function damped_non_linear_driven(ddu, du, u, p, t)
    #ddu[1] = -2 * κ * du[1] - E_J * C_inv *sin(u[1] / ϕ_quantum) / ϕ_quantum + A * cos(ω * t)
    ddu[1] = -2 * κ * du[1] - E_J * C_inv *u[1] / ϕ_quantum / ϕ_quantum + A * cos(ω* t) #linear model
end

A=10
# Initial conditions
ϕ_0 = [1.]
ϕ_dot_0 = [1.]

κ = 1 / 20 # GHZ Damping rate
C_inv = 0.147 * 2 * pi # ħ/e^2 in GHz

E_J = 90 *2pi /2  # ħ * GHz
ϕ_quantum = 1/2 # ħ/e 
ω_nl = sqrt(4*C_inv*E_J)
Z_nl=C_inv/ω_nl

# Drive 
ω = ω_nl

tspan = (0.0, 1.0/κ) # Time span for the simulation
dt = 1 / (60 * ω_nl) # Time step for the simulation

prob = SecondOrderODEProblem(damped_non_linear_driven, ϕ_dot_0, ϕ_0, tspan)
sol = solve(prob, VelocityVerlet(), dt=dt)
t= sol.t
# Extract solution arrays
ϕ_dot= [u[1] for u in sol.u]
ϕ = [u[2] for u in sol.u]
q = ϕ_dot / C_inv;


In [ ]:
Ω=sqrt(ω_nl^2-κ^2)
ϕ_p_ana = @. A *(ω_nl^2-ω^2)/((ω_nl^2-ω^2)^2+(2 * κ * ω_nl)^2)*cos(ω_nl * t) +
            A * 2 * κ * ω_nl/((ω_nl^2-ω^2)^2+(2 * κ * ω_nl)^2)*sin(ω_nl * t)
ϕ_dot_0_h = ϕ_dot_0[1] - A * 2 * κ * ω_nl/((ω_nl^2-ω^2)^2+(2 * κ * ω_nl)^2) * ω_nl
ϕ_0_h = ϕ_0[1] - A * (ω_nl^2-ω^2)/((ω_nl^2-ω^2)^2+(2 * κ * ω_nl)^2)
ϕ_h_ana = @. exp(-κ*t)*(ϕ_0_h*cos(Ω*t) + (ϕ_dot_0_h+κ*ϕ_0_h)*sin(Ω*t)/Ω)

ϕ_ana_tot = ϕ_p_ana + ϕ_h_ana;

In [ ]:
plot(t*κ, ϕ)
plot!(t*κ,ϕ_ana_tot, label="ϕ_h_ana", legend=:topright)
#plot!(xlims=(4.0,4.1))

In [ ]:
plot(t*κ, (ϕ-ϕ_ana_tot))

In [ ]:

fs = length(t)/(maximum(t)-minimum(t)) # sampling frequency
freqs = fftshift(fftfreq(length(t), fs))

F = fftshift(fft(ϕ))
F_ana = fftshift(fft(ϕ_ana_tot))

plot(freqs, abs.(F), label="ϕ", xlim=(0, 10), legend=:topright)
plot!(freqs, abs.(F_ana), label="ϕ_ana", legend=:topright)
vline!(freqs, [ω_nl/2pi], label="ω_nl", color=:red, linestyle=:dash)
plot!(xlims=(2,6), yaxis=:log, legend=:bottomright)